# Chapter 11 — Sandbox the Code

**Secure Agentic AI Developer Workbook**

> OWASP: ASI05 (Unexpected Code Execution)

No API key required — mock exercises only.


In [ ]:
%pip install -q colorama


In [ ]:
from colorama import Fore, init
init(autoreset=True)
def ok(m): print(Fore.GREEN + '✅ ' + m)
def bad(m): print(Fore.RED + '💥 ' + m)
ok('Setup complete.')


## Exercise 1 — exec exfiltrates a secret


In [ ]:
FAKE = {"OPENAI_API_KEY": "sk-prod-secret"}
def run_model_code(code: str) -> str:
    local = {}
    exec(code, {"__builtins__": __builtins__, "env": FAKE}, local)
    return str(local.get("result", ""))
stolen = run_model_code('result = env["OPENAI_API_KEY"]')
bad(f"Secret leaked via exec: {stolen}")


## Exercise 2 — AST calculator blocks imports


In [ ]:
import ast
ALLOWED = (ast.Expression, ast.BinOp, ast.UnaryOp, ast.Constant, ast.Add, ast.Sub, ast.Mult, ast.Div, ast.USub)
def safe_calc(expr: str) -> float:
    tree = ast.parse(expr, mode="eval")
    for n in ast.walk(tree):
        if not isinstance(n, ALLOWED):
            raise ValueError(type(n).__name__)
    return float(eval(compile(tree, "<c>", "eval"), {"__builtins__": {}}, {}))
ok(f"2+3*4 = {safe_calc('2+3*4')}")
try:
    safe_calc("__import__('os').system('id')")
    bad("should have blocked")
except ValueError as e:
    ok(f"Blocked dangerous expr ({e})")


## Your turn — implement safe_calc_your_fix


In [ ]:
def safe_calc_your_fix(expr: str) -> float:
    """TODO: copy the AST allowlist approach from Exercise 2."""
    raise NotImplementedError("implement safe_calc_your_fix")

# Uncomment when ready:
# assert safe_calc_your_fix("1+2") == 3.0
# try:
#     safe_calc_your_fix("__import__('os').getcwd()")
#     raise AssertionError("should block")
# except ValueError:
#     pass
# ok("Your patch passed")

